In [1]:
print("Youtube project started here ....")

Youtube project started here ....


# URLS

In [2]:
URLS = ['https://www.youtube.com/watch?v=Pn95eOlw5qk&t=5s' ,
        'https://www.youtube.com/watch?v=KNwMiydCYA4', 
        'https://www.youtube.com/watch?v=S7TUe5w6RHo']

# Imports

In [6]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings , OllamaLLM
from joblib import load , dump
from langchain_core.prompts import PromptTemplate

# Constants

In [22]:
from sentence_transformers import CrossEncoder

In [23]:
EMBEDDINGS = OllamaEmbeddings(model="bge-m3")
LLM = OllamaLLM(model= "qwen3:1.7b")
Reranking_model = CrossEncoder(model_name_or_path = 'cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

# Assign chunk ids..

In [5]:
def assign_ids(chunks):
    for index , chunk in enumerate(chunks):
        chunk.metadata['chunk_id'] = f"chunk_No:{index}"

# Save chunks in a separate file..

In [6]:
def save_chunks(chunks , vid_name):
    dump(value = chunks , filename = vid_name)

# Build eval dataset.

# Dataset generation prompt

In [7]:
Dataset_Generation_Prompt = PromptTemplate(
template="""
You are an expert in creating evaluation datasets for Retrieval-Augmented Generation (RAG) systems.

Generate EXACTLY 10 diverse evaluation examples using ONLY the provided context.

IMPORTANT REQUIREMENT:

* Every question MUST require information from MULTIPLE DISTINCT chunks to answer correctly.
* Each question MUST have at least 2 relevant_chunk_ids.
* Prefer questions that require combining, connecting, comparing, or reasoning across information from 2 or more chunks.
* Do NOT generate questions that can be completely answered from a single chunk.
* The ground_truth_answer must depend on information from ALL listed relevant chunks.

RULES:

* Use ONLY information explicitly present in the context. Do NOT use external knowledge, do NOT invent facts.
* Copy chunk IDs EXACTLY as they appear in the context. Do NOT invent, modify, shorten, or reformat them.
* Every chunk ID you output MUST exist in the provided context.
* Each question MUST have at least 2 relevant_chunk_ids.
* Questions must be diverse: mix factual, conceptual, how, why, comparison, cause-and-effect, and relationship questions.
* Avoid duplicate or overly similar questions.
* Avoid simple yes/no questions.
* Avoid questions that can be answered using only one chunk.
* Avoid including irrelevant chunks just to increase the number of relevant chunks.
* relevant_chunk_ids must include ONLY chunks that are directly necessary to construct the answer.
* ground_truth_answer must be concise, complete, and fully supported by ALL relevant chunks.
* If a question can be answered using only one of the listed chunks, DO NOT use that question.

MULTI-CHUNK QUESTION GUIDELINES:
Good questions should require combining information such as:

* Information from chunk A + information from chunk B.
* A fact from one chunk and its explanation/consequence from another chunk.
* Comparing information presented in different chunks.
* Connecting events, concepts, technologies, or causes described in different chunks.
* Understanding a relationship between information distributed across multiple chunks.

Bad example:

* Question can be answered entirely from chunk A.
* relevant_chunk_ids: ["chunk_A", "chunk_B"]

Good example:

* The answer requires information from both chunk A and chunk B.
* relevant_chunk_ids: ["chunk_A", "chunk_B"]

OUTPUT FORMAT — CRITICAL:
Return ONLY a valid JSON array. Nothing else.

* No markdown, no ```json fences, no explanations, no headings, no comments.
* No text before the opening [ or after the closing ].
* The array MUST contain EXACTLY 10 objects.
* Each object MUST have EXACTLY these 3 fields, spelled exactly like this:
  "question" (string)
  "ground_truth_answer" (string)
  "relevant_chunk_ids" (array of strings)

Example structure (do not copy the values, only the shape):
[
{{
"question": "A question requiring information from multiple chunks",
"ground_truth_answer": "An answer combining information from multiple chunks",
"relevant_chunk_ids": ["<exact_id_from_context>", "<exact_id_from_context>"]
}}
]

Before answering, internally verify:

1. Output starts with [ and ends with ].
2. There are exactly 10 objects.
3. Every question requires information from at least 2 distinct chunks.
4. Every relevant_chunk_ids array contains at least 2 chunk IDs.
5. Every relevant_chunk_ids value exists verbatim in the context.
6. Every listed chunk directly contributes necessary information to the answer.
7. No question can be completely answered using only one relevant chunk.
8. No text exists outside the JSON array.

Context:
{context}
""",
input_variables=["context"]
)


# Create the context

In [8]:
def get_context(chunks_file_path):

    chunks = load(filename = chunks_file_path)

    context = ""

    for chunk in chunks:
        context += "\n" + chunk.metadata['chunk_id'] + "\n" + chunk.page_content


    return context    

In [9]:
def get_ready_prompt(context):
    
    Ready_prompt = Dataset_Generation_Prompt.invoke(input= {
            'context' : context
    })

    return Ready_prompt.text.strip()    


In [10]:
def chunk_documents(documents):

    chunker = SemanticChunker(embeddings = EMBEDDINGS , 
                              breakpoint_threshold_type = 'standard_deviation' , 
                              breakpoint_threshold_amount = 1) 

    chunks = chunker.split_documents(documents = documents)

    return chunks

In [11]:
def load_documents(youtube_url):

    loader = YoutubeLoader.from_youtube_url(youtube_url = youtube_url , language = "en-US")

    documents = loader.load()

    return documents

In [12]:
def build_ready_prompt(youtube_url , chunk_file_path):

    documents = load_documents(youtube_url= youtube_url)

    chunks = chunk_documents(documents= documents)

    assign_ids(chunks = chunks)

    save_chunks(chunks = chunks , vid_name = chunk_file_path)
    
    context = get_context(chunks_file_path = chunk_file_path)

    ready_prompt = get_ready_prompt(context= context)

    return ready_prompt

# Draft "How to test rag system"

# Lets start with video1

In [7]:
import warnings
warnings.filterwarnings(action='ignore')
from langchain_chroma import Chroma

In [10]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever


In [9]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [8]:
import json
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [11]:
def Rerank(reranker_model , query , docs , Top_k):
    reranked_documents = []
    pairs = []
    for doc in docs: 
        pair = (query , doc.page_content)
        pairs.append(pair)

    scores = reranker_model.predict(inputs = pairs)

    docs_scores = list(zip(scores , docs))

    sorted_result = sorted(docs_scores , reverse = True)

    for _ , doc in sorted_result[:Top_k]:
        reranked_documents.append(doc)

    return reranked_documents

In [12]:
import json
def build_retrieved_generated_dataset(Evaluation_dataset_file_path , vector_DB , output_file):
    # Load dataset.
    with open(Evaluation_dataset_file_path , "r" , encoding="utf-8") as f:
        dataset = json.load(f)

    recalls = []    


    for item in dataset:
        retrieved_chunks = []
        print('question : ' , item['question'])
        print('relevant_chunk_ids : ' , item['relevant_chunk_ids'])

        retriever = vector_DB.as_retriever(
        search_type = "mmr",
        search_kwargs = {
            "k": len(item['relevant_chunk_ids']) , 
            'fetch_k' : len(item['relevant_chunk_ids']) * 2, 
            'lambda_mult' : 0.5
        })


        keywords_search_retriever = BM25Retriever.from_documents(documents = load(filename= "D:/Youtube rag system/Chunks/chunks_file2"))
        keywords_search_retriever.k = len(item['relevant_chunk_ids'])


        hybrid_search_retriever = EnsembleRetriever(retrievers= [retriever , keywords_search_retriever], weights= [0.5 , 0.5])
        docs = hybrid_search_retriever.invoke(input = item['question'])

        reranked_docs = Rerank(reranker_model = Reranking_model , query = item['question'] , docs = docs , Top_k = len(item['relevant_chunk_ids']))
        for doc in reranked_docs :
            retrieved_chunks.append(doc.metadata['chunk_id'])

        """
        prompt = PromptTemplate(
            template="""#Answer the question using only the provided context.

            #Context:
            #{context}

            #Question:
            #{question}

            #Answer:
            #"""
            #input_variables=['question', 'context']
        #)
        #context = ""

        #for doc in result :
        #    context += "\n" + doc.page_content 

        #ready_prompt = prompt.invoke(input={'context' : context , 'question' : item['question']})

        #generated_answer = LLM.invoke(input = ready_prompt)

        #retrieved_contexts = [doc.page_content for doc in result]
                
        print("retrieved_chunks : " , retrieved_chunks)    

        #print("generated answer : " , generated_answer)

        #print("Ground truth answer : " , item['ground_truth_answer'])

        #print("retrieved_contexts : " , retrieved_contexts)

        #item['response'] = generated_answer
        #item['retrieved_contexts'] = retrieved_contexts
        #item['retrieved_chunk_ids'] = retrieved_chunks
        #"""
        recall = calc_recall_precesion_f1_score(relevant_chunks= item['relevant_chunk_ids'] , retrieved_chunks= retrieved_chunks)
        recalls.append(recall)

        print("*" * 80)
        print("\n\n")
    print("Total recall is : " , sum(recalls) / len(recalls))    


    with open(file= output_file , mode='w' , encoding='utf-8') as f :
        json.dump(dataset , f , ensure_ascii= False , indent= 4)

In [13]:
def calc_recall_precesion_f1_score(relevant_chunks , retrieved_chunks):
    intersection = set(relevant_chunks) & set(retrieved_chunks)

    if len(intersection) != 0 : 
        recall = len(intersection) / len(relevant_chunks) * 100 
        precesion = len(intersection) / len(retrieved_chunks) * 100
        f1_score = 2 * (recall * precesion) / (recall + precesion)
        print("recall : " , recall,"%")
        print("precesion : " , precesion,"%")
        print("f1_score : " , f1_score,"%")

    else :
        precesion = 0 
        recall = 0 
        f1_score = 0    

    return recall     

# FULL PIPELINE (EVAL DATASET BUILDER)

In [14]:
from Ingestion.Vector_DB import load_vector_DB , creat_vector_DB
from Ingestion.chunker import chunk_documents
from Ingestion.youtube_loader import load_documents
from Ingestion.utils import assign_ids , save_chunks , get_context , get_ready_prompt , Dataset_Generation_Prompt

In [7]:
config = {'llm' : OllamaLLM(model= 'qwen3:1.7b') , 
          'embeddings' : OllamaEmbeddings(model= 'bge-m3')}

In [22]:
from Eval_dataset_generator.Building_ready_prompt import build_ready_prompt

In [ ]:
for index , url in enumerate(URLS):
    ready_prompt = build_ready_prompt(youtube_url = url, chunk_file_path = f"Chunks/chunks_file{index + 1}" , embeddings = config['embeddings'])
    print("ready prompt : " , ready_prompt)

ready prompt :  You are an expert in creating evaluation datasets for Retrieval-Augmented Generation (RAG) systems.

Generate EXACTLY 10 diverse evaluation examples using ONLY the provided context.

RULES:
- Use ONLY information explicitly present in the context. Do NOT use external knowledge, do NOT invent facts.
- Copy chunk IDs EXACTLY as they appear in the context. Do NOT invent, modify, shorten, or reformat them.
- Every chunk ID you output MUST exist in the provided context.
- Questions must be diverse: mix factual, conceptual, how, why, comparison, and relationship questions.
- Avoid duplicate or overly similar questions.
- Avoid simple yes/no questions.
- ground_truth_answer must be concise, complete, and fully supported by the context. Do NOT hallucinate.
- relevant_chunk_ids must include ONLY chunks that directly support the answer (not every chunk that mentions the same topic).

OUTPUT FORMAT — CRITICAL:
Return ONLY a valid JSON array. Nothing else.
- No markdown, no ```json

In [ ]:
vid1_chunks = load(filename= "Chunks/chunks_file1")
vid2_chunks = load(filename= "Chunks/chunks_file2")
vid3_chunks = load(filename= "Chunks/chunks_file3")

In [18]:
vector_DB1 = creat_vector_DB(collection_name="vid1_collection" , 
                persist_directory= "vid1_DB" , 
                embedding_function= config['embeddings'] ,
                chunks = vid1_chunks)

vector_DB2 = creat_vector_DB(collection_name="vid2_collection" , 
                persist_directory= "vid2_DB" , 
                embedding_function= config['embeddings'] ,
                chunks = vid2_chunks)

vector_DB3 = creat_vector_DB(collection_name="vid3_collection" , 
                persist_directory= "vid3_DB" , 
                embedding_function= config['embeddings'] ,
                chunks = vid3_chunks)

NameError: name 'vid1_chunks' is not defined

In [19]:
DB_VID1 = load_vector_DB(persist_directory = 'D:/Youtube rag system/Databases/vid1_DB' , 
               collection_name = 'vid1_collection', 
               embeddings = config['embeddings'])

DB_VID2 = load_vector_DB(persist_directory = 'D:/Youtube rag system/Databases/vid2_DB' , 
               collection_name = 'vid2_collection', 
               embeddings = config['embeddings'])

DB_VID3 = load_vector_DB(persist_directory = 'D:/Youtube rag system/Databases/vid3_DB' , 
               collection_name = 'vid3_collection', 
               embeddings = config['embeddings'])

# reranking => [86.66 , 86.66 , 80.0] but still important to catch only the important informations...
# without reranking => [ 86.66 , 86.66 , 80.0]

In [27]:
build_retrieved_generated_dataset(Evaluation_dataset_file_path = "Evaluation/Eval_dataset3.json", vector_DB = DB_VID3 , output_file= 'Evaluation/Ragas_Eval_dataset3.json')

question :  How did Earth's atmosphere and conditions evolve from its formation through the early Hadean and Archean eons?
relevant_chunk_ids :  ['chunk_No:1']
retrieved_chunks :  ['chunk_No:1']
recall :  100.0 %
precesion :  100.0 %
f1_score :  100.0 %
********************************************************************************



question :  What role did the Moon's formation and the heavy asteroid bombardment play in early Earth history?
relevant_chunk_ids :  ['chunk_No:1']
retrieved_chunks :  ['chunk_No:1']
recall :  100.0 %
precesion :  100.0 %
f1_score :  100.0 %
********************************************************************************



question :  How did the transition to oxygen-rich atmosphere unfold from the Great Oxidation Event through the Rhyacian and Orosirian periods?
relevant_chunk_ids :  ['chunk_No:4']
retrieved_chunks :  ['chunk_No:4']
recall :  100.0 %
precesion :  100.0 %
f1_score :  100.0 %
**************************************************************

In [6]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever


In [10]:
DB_VID1 = load_vector_DB(persist_directory = 'D:/Youtube rag system/vid1_DB' , 
               collection_name = 'vid1_collection', 
               embeddings = EMBEDDINGS)

In [11]:
similarity_retriever = DB_VID1.as_retriever(search_type = "similarity" , 
                        search_kwargs = {'k' : 3})

In [12]:
docs1 = load(filename="D:/Youtube rag system/Chunks/chunks_file1")
docs1

[Document(metadata={'source': 'Pn95eOlw5qk', 'chunk_id': 'chunk_No:0'}, page_content='Okay, so here\'s a question for you. What if you could hire a team of brilliant assistants? Assistants who never sleep, never get tired, never ask for a raise, and just hand them a goal and say, "Get it done." No step-by-step instructions, no hand-holding, just results. That\'s not science fiction anymore.'),
 Document(metadata={'source': 'Pn95eOlw5qk', 'chunk_id': 'chunk_No:1'}, page_content="That's agentic AI. And by the end of this video, you're going to understand exactly what it is, how it works, and why every single person in tech right now is talking about it. We're going to cover everything from the absolute basics all the way to vector databases, RAG, MCP, multi-agent systems, and real architectures being used in production today. So, whether you're a complete beginner or someone who already knows a bit about AI and just wants things to finally click, this video is for you."),
 Document(metad

In [13]:
keywords_search_retriever = BM25Retriever.from_documents(documents = docs1)

In [14]:
ensemble_retriever = EnsembleRetriever(retrievers = [similarity_retriever , keywords_search_retriever] ,
                                        weights = [0.5 , 0.5])

In [15]:
result = ensemble_retriever.invoke(input = "What does it mean for an AI agent to have 'autonomy'?")

In [51]:
# reranker build

from sentence_transformers import CrossEncoder


reranker_model = CrossEncoder(model_name_or_path= "cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

# Reranking method

In [76]:
query = "What are the four steps of the core AI agent loop?"

In [77]:
result = ensemble_retriever.invoke(input = query)


In [114]:
Rerank(reranker_model= reranker_model , query= query , docs= result , Top_k=3)

[Document(id='5692812b-1753-4ab1-8734-a1cdc1a02354', metadata={'source': 'Pn95eOlw5qk', 'chunk_id': 'chunk_No:16'}, page_content="It can think through problems, break them into pieces, and figure out the best approach. Third, planning. It doesn't just react to the moment, it creates multi-step plans and adjusts them as things change. Fourth, action. It can actually do things in the real world, call APIs, run code, send emails, search the web. And fifth, adaptation. It learns from feedback, from its own mistakes, from what works and what doesn't. A regular chatbot has maybe the first two partially. An agent has all five, and that's a massive difference in what it can actually accomplish. Part four, the core agent loop, how it actually runs. Here's something I want you to burn into your brain, because this is the foundation of everything else. Every single AI agent, no matter how simple or complex, runs on a loop, a continuous cycle. Step one, perceive."),
 Document(metadata={'source': '

In [28]:
Database_path = "D:/Youtube rag system/Datasets/new_vid_6jtjD8323qs"

In [30]:
vector_DB_test = load_vector_DB(persist_directory="D:/Youtube rag system/Datasets/new_vid_6jtjD8323qs" , 
               collection_name = "new_vid_6jtjD8323qs",
               embeddings= config['embeddings'])

In [34]:
len(vector_DB_test.get(include = ['documents'])['documents'])

1

In [3]:
from Ingestion.chunker import chunk_documents
from Ingestion.youtube_loader import load_documents , load_documents_v2
docs = load_documents_v2(youtube_url= "https://www.youtube.com/watch?v=6jtjD8323qs&t=137s")

VIDEO ID: 6jtjD8323qs
AVAILABLE TRANSCRIPTS:
language: ar | Arabic (auto-generated) | generated: True
SELECTED TRANSCRIPT: ar
TRANSCRIPT LENGTH: 32311


In [8]:
chunks = chunk_documents(
    documents=docs,
    embeddings=config["embeddings"],
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=1
)

print("NUMBER OF CHUNKS:", len(chunks))

for i, chunk in enumerate(chunks):
    print(
        f"Chunk {i} -> {len(chunk.page_content)} characters"
    )

NUMBER OF CHUNKS: 1
Chunk 0 -> 32311 characters


In [1]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()

embeddings = HuggingFaceEndpointEmbeddings(
    model="intfloat/multilingual-e5-large",
    huggingfacehub_api_token=os.getenv("HF_TOKEN")
)

embedding = embeddings.embed_query(
    "What is Retrieval Augmented Generation?"
)

print("Embedding length:", len(embedding))
print("First 5 values:", embedding[:5])

Embedding length: 1024
First 5 values: [0.03377922624349594, -0.0033992144744843245, -0.03706052899360657, -0.0332348495721817, 0.025312501937150955]


In [2]:
arabic_embedding = embeddings.embed_query(
    "ما هو الهدف من استخدام نظام RAG؟"
)

english_embedding = embeddings.embed_query(
    "What is the purpose of using a RAG system?"
)

print("Arabic length:", len(arabic_embedding))
print("English length:", len(english_embedding))

Arabic length: 1024
English length: 1024


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    [arabic_embedding],
    [english_embedding]
)[0][0]

print("Arabic-English similarity:", similarity)

Arabic-English similarity: 0.9316114169625324
